# CARE H5 Dataset Visualizer
Interactive visualization of paired low/high SNR patches from CARE training data

In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, Dropdown, VBox, HBox, Output, Label
import ipywidgets as widgets
from IPython.display import display, clear_output

In [ ]:
# Set the path to your CARE H5 file
H5_PATH = "/path/to/care_training_data.h5"

In [ ]:
# Open H5 file and inspect structure
h5f = h5py.File(H5_PATH, 'r')

print("H5 File Structure:")
print(f"Root keys: {list(h5f.keys())}")
for split in h5f.keys():
    print(f"\n{split}/:")
    for key in h5f[split].keys():
        ds = h5f[split][key]
        print(f"  {key}: {ds.shape}, dtype={ds.dtype}, "
              f"min={ds[0].min():.4f}, max={ds[0].max():.4f}")

In [ ]:
class CareH5Visualizer:
    def __init__(self, h5_file):
        self.h5f = h5_file
        self.current_split = 'train'

        self.low = self.h5f[self.current_split]['low']
        self.high = self.h5f[self.current_split]['high']

        self.n_samples = self.low.shape[0]
        self.n_z = self.low.shape[1]

        self.output = Output()
        self.stats_output = Output()

    def switch_split(self, split):
        self.current_split = split
        self.low = self.h5f[split]['low']
        self.high = self.h5f[split]['high']
        self.n_samples = self.low.shape[0]
        self.n_z = self.low.shape[1]

    def visualize(self, sample_idx, z_idx, show_diff):
        with self.output:
            clear_output(wait=True)

            low_patch = self.low[sample_idx]
            high_patch = self.high[sample_idx]

            low_slice = low_patch[z_idx]
            high_slice = high_patch[z_idx]

            ncols = 3 if show_diff else 2
            fig, axes = plt.subplots(1, ncols, figsize=(5 * ncols, 5))

            vmin = min(low_slice.min(), high_slice.min())
            vmax = max(low_slice.max(), high_slice.max())

            axes[0].imshow(low_slice, cmap='gray', vmin=vmin, vmax=vmax)
            axes[0].set_title(f'Low SNR\nSample {sample_idx}, Z={z_idx}')
            axes[0].axis('off')

            axes[1].imshow(high_slice, cmap='gray', vmin=vmin, vmax=vmax)
            axes[1].set_title(f'High SNR\nSample {sample_idx}, Z={z_idx}')
            axes[1].axis('off')

            if show_diff:
                diff = high_slice.astype(np.float32) - low_slice.astype(np.float32)
                im = axes[2].imshow(diff, cmap='RdBu_r')
                axes[2].set_title(f'Difference (High - Low)\nZ={z_idx}')
                axes[2].axis('off')
                plt.colorbar(im, ax=axes[2], fraction=0.046)

            plt.tight_layout()
            plt.show()

        with self.stats_output:
            clear_output(wait=True)
            print("=" * 60)
            print(f"Patch {sample_idx} — {self.current_split} split")
            print("=" * 60)
            print(f"Shape: {low_patch.shape}")
            print(f"Low  — min: {low_patch.min():.4f}, max: {low_patch.max():.4f}, "
                  f"mean: {low_patch.mean():.4f}, std: {low_patch.std():.4f}")
            print(f"High — min: {high_patch.min():.4f}, max: {high_patch.max():.4f}, "
                  f"mean: {high_patch.mean():.4f}, std: {high_patch.std():.4f}")
            mse = np.mean((low_patch.astype(np.float32) - high_patch.astype(np.float32)) ** 2)
            print(f"MSE(low, high): {mse:.6f}")
            print("=" * 60)

    def create_widgets(self):
        split_dropdown = Dropdown(
            options=list(self.h5f.keys()),
            value=self.current_split,
            description='Split:'
        )

        sample_slider = IntSlider(
            min=0, max=self.n_samples - 1, step=1, value=0,
            description='Sample:',
            continuous_update=False
        )

        z_slider = IntSlider(
            min=0, max=self.n_z - 1, step=1, value=self.n_z // 2,
            description='Z slice:',
            continuous_update=False
        )

        diff_checkbox = widgets.Checkbox(
            value=True,
            description='Show Difference'
        )

        def on_split_change(change):
            self.switch_split(change['new'])
            sample_slider.max = self.n_samples - 1
            sample_slider.value = min(sample_slider.value, self.n_samples - 1)
            z_slider.max = self.n_z - 1
            z_slider.value = min(z_slider.value, self.n_z - 1)
            self.visualize(sample_slider.value, z_slider.value, diff_checkbox.value)

        split_dropdown.observe(on_split_change, names='value')

        def update(sample_idx, z_idx, show_diff):
            self.visualize(sample_idx, z_idx, show_diff)

        interactive_widget = widgets.interactive(
            update,
            sample_idx=sample_slider,
            z_idx=z_slider,
            show_diff=diff_checkbox,
        )

        controls = VBox([
            Label(f'Dataset: {self.n_samples} patches, shape: {self.low.shape}'),
            split_dropdown,
            sample_slider,
            z_slider,
            diff_checkbox,
        ])

        display(controls)
        display(self.output)
        display(self.stats_output)

        self.visualize(0, self.n_z // 2, True)

In [ ]:
# Create and display the visualizer
viz = CareH5Visualizer(h5f)
viz.create_widgets()